# Hyperparameter Tuning — SVM with Bayesian Search

In [2]:
import pandas as pd
import numpy as np
import sys, io, glob, os
import matplotlib
import pickle

# Add Feature_Selection folder to path
sys.path.append('../5.Feature_Selection/')
matplotlib.use('Agg')

# Suppress noisy output when importing notebooks
old_stdout = sys.stdout
sys.stdout = io.StringIO()
from importnb import Notebook
with Notebook():
    import custom_features, feauture_selection
sys.stdout = old_stdout

# Clean up temporary feature files created by feature_selection
for f in glob.glob("*_features_*.npz"):
    os.remove(f)


/Users/negin/Desktop/untitled folder/7.Model_performances/../5.Feature_Selection/feauture_selection.ipynb:607: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/negin/Desktop/untitled folder/7.Model_performances/../5.Feature_Selection/feauture_selection.ipynb:646: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/negin/Desktop/untitled folder/7.Model_performances/../5.Feature_Selection/feauture_selection.ipynb:607: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/negin/Desktop/untitled folder/7.Model_performances/../5.Feature_Selection/feauture_selection.ipynb:646: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/negin/Desktop/untitled folder/7.Model_performances/../5.Feature_Selection/feauture_selection.ipynb:607: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/Users/neg

In [3]:
print(feauture_selection.features_to_use)


In [4]:
def update_vonheijne(sets, matrix):
    seq_features = []
    for seq in sets:
        seq = seq.replace("X", "").replace("U", "C")
        vonhejine = custom_features.vonheijne_feature(matrix, seq)
        seq_features.append(vonhejine)
    return np.array(seq_features)


In [5]:
# Load train / test / validation splits (iteration 5)
loaded_data_train = np.load('../5.Feature_Selection/training_features_5.npz')
x_train  = loaded_data_train['matrix']
y_train  = loaded_data_train['target']

loaded_data_test = np.load('../5.Feature_Selection/testing_features_5.npz')
x_test   = loaded_data_test['matrix']
y_test   = loaded_data_test['target']

loaded_data_validation = np.load('../5.Feature_Selection/validation_features_5.npz')
x_validation = loaded_data_validation['matrix']
y_validation = loaded_data_validation['target']

# Concatenate training + test for final training matrix
x_training_conc = np.concatenate((x_train, x_test), axis=0)
y_training_conc = np.concatenate((y_train, y_test), axis=0)


In [7]:
# Load benchmark set and re-encode features
dataset  = pd.read_csv("../2.Data_Preparation/train_bench.tsv", sep="\t")
benchmark = dataset.query("Set=='Benchmark'")
training = pd.concat([
    dataset.query("Set=='2'"),
    dataset.query("Set=='3' or Set=='4' or Set=='5'")
    # Set 1 excluded — it was the validation fold in iteration 5
    # and is NOT part of x_training_conc
], axis=0, ignore_index=True)
matrix_training = custom_features.get_pswm(training, 13, 2)

# Update Von Heijne feature in training matrix using the new PSWM
x_training_conc[:, 17] = update_vonheijne(training["Sequence"], matrix_training)

# FIX 1: variable was named x_benchmark (undefined); the correct variable
# returned by get_all_features is feature_set_benchmark
feature_set_benchmark, feature_order_training = custom_features.get_all_features(
    benchmark["Sequence"], matrix_training, 15
)

vector_neg_pos = benchmark["Class"]
target_benchmark_vector = vector_neg_pos.map({"Positive": 1, "Negative": 0}).to_numpy()


In [8]:
# Save feature matrices for later use
np.savez('benchmark_features.npz',
         matrix=feature_set_benchmark,       # FIX 1: was x_benchmark
         target=target_benchmark_vector)
np.savez('training_features.npz',
         matrix=x_training_conc, target=y_training_conc)
np.savez('validation_features.npz',
         matrix=x_validation, target=y_validation)


## Hyperparameter Tuning

In [9]:
import numpy as np
from sklearn.metrics import matthews_corrcoef, make_scorer   # FIX 2: import make_scorer
# !pip install scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Categorical
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix


In [10]:
pipeline = Pipeline([("scaler", StandardScaler()), ("svm", SVC(cache_size=1500))])


In [11]:
search_space = {
    "svm__kernel": Categorical(["rbf"]),
    "svm__C":     Real(0.01, 100, prior="log-uniform"),
    "svm__gamma": Real(0.01, 100, prior="log-uniform"),
}


In [12]:
# FIX 2: scoring="matthews_corrcoef" is not a valid string scorer for
# BayesSearchCV — replaced with make_scorer(matthews_corrcoef).
bayes = BayesSearchCV(
    estimator=pipeline,
    search_spaces=search_space,
    scoring=make_scorer(matthews_corrcoef),   # <── fixed
    n_jobs=-1,
    refit=False,
    random_state=42,
    verbose=2,
    cv=5,
    n_iter=60
)
bayes.fit(x_training_conc, y_training_conc)

print("\n[Best parameters found:]")
print(bayes.best_params_)
print(f"[Best MCC validation] {bayes.best_score_:.4f}")


In [13]:
# Predict the benchmark set
pipeline.set_params(**bayes.best_params_).fit(x_training_conc, y_training_conc)

# FIX 1: was x_benchmark (undefined) — correct variable is feature_set_benchmark
bench_pred = pipeline.predict(feature_set_benchmark)
mcc_bayes  = matthews_corrcoef(target_benchmark_vector, bench_pred)
print(f"MCC on benchmark set (Bayesian search): {mcc_bayes}")


In [14]:
conf_mat = confusion_matrix(target_benchmark_vector, bench_pred)
print(conf_mat)


Save the model for later use

In [15]:
with open('SignalPeptideSVM.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
